# 🧹 Modul 02: Data Cleaning & Preprocessing (Jantung Penambangan Data)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antonprafanto/data-mining-zero-to-hero/blob/main/notebooks/02_Data_Cleaning_and_Preprocessing.ipynb)

> **Repositori Resmi**: [github.com/antonprafanto/data-mining-zero-to-hero](https://github.com/antonprafanto/data-mining-zero-to-hero)  
> **Tingkat Kesulitan**: Pemula Menengah (From Zero to Hero)  
> **Estimasi Waktu Belajar**: 55 – 70 Menit (Santai dan interaktif)  
> **Tujuan Modul**: Memahami prinsip *Garbage In Garbage Out*, mendeteksi dan mengisi data kosong (*Missing Values: MCAR, MAR, MNAR, KNNImputer*), mendeteksi dan menangani pencilan (*Outliers: IQR & Z-Score*), standarisasi skala (*MinMaxScaler, StandardScaler, RobustScaler*), serta pengkodean kategori (*Ordinal vs One-Hot Encoding*).

---

## 💡 1. Kenapa Preprocessing Adalah Jantung Data Mining?

> *"80% waktu seorang praktisi data dihabiskan untuk membersihkan dan menyiapkan data, sedangkan melatih modelnya sendiri hanya butuh 20% waktu."*

Ada pepatah legendaris dalam dunia komputer:
### 🗑️ **"Garbage In, Garbage Out" (Sampah Masuk, Sampah Pula yang Keluar)**
Jika Anda menyuapi algoritma tercanggih di dunia dengan data yang kotor, bolong-bolong, salah ketik, dan penuh angka aneh, maka hasil prediksi atau rekomendasinya pasti akan **menyesatkan dan membahayakan keputusan bisnis**.

Mari kita pelajari 4 pilar utama pembersihan data secara bertahap dan mudah dipahami!

---

## 🩹 2. Penanganan Nilai Hilang (*Missing Values*)

Di dunia nyata, data sering kali bolong/kosong (bernilai `NaN` atau *Not a Number*). Mengapa data bisa hilang?
- Pelanggan lupa mengisi formulir pendaftaran.
- Sensor alat kesehatan sempat mati sesaat.
- Sistem database mengalami gangguan saat proses penyimpanan.

### 📖 3 Jenis Missing Values Menurut Literatur Akademik:
1. **MCAR (*Missing Completely at Random*)**: Data hilang murni karena kebetulan acak (misal: lembar kuesioner tertiup angin dan hilang tanpa pola).
2. **MAR (*Missing at Random*)**: Data hilang berkaitan dengan variabel lain yang tercatat (misal: pria lebih jarang mengisi kolom berat badan dibanding wanita).
3. **MNAR (*Missing Not at Random*)**: Data hilang justru karena nilai dari variabel itu sendiri (misal: nasabah yang penghasilannya sangat tinggi sengaja mengosongkan kolom penghasilan karena privasi).

Mari kita buat dataset simulasi nasabah bank yang realistis dan kotor:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Simulasi data nasabah pengajuan kredit perbankan
data_kotor = {
    'ID_Nasabah': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'Usia': [25, np.nan, 28, 45, 120, 32, 29, np.nan, 50, 36],             # Ada NaN dan Outlier 120 tahun
    'Gaji_Juta': [5.0, 7.5, np.nan, 15.0, 12.0, 8.0, 6.5, 4.5, 95.0, 11.0],  # Ada NaN dan Outlier 95 Juta
    'Tingkat_Pendidikan': ['S1', 'D3', 'SMA', 'S2', 'S1', 'S1', 'D3', 'SMA', 'S2', np.nan],
    'Status_Pernikahan': ['Lajang', 'Menikah', 'Lajang', np.nan, 'Menikah', 'Lajang', 'Menikah', 'Lajang', 'Menikah', 'Menikah'],
    'Skor_Kredit': [650, 710, 590, 780, 680, 620, np.nan, 580, 800, 720]
}

df = pd.DataFrame(data_kotor)
print("=== DATASET MENTAH NASABAH (MASIH KOTOR) ===")
df

### 🔍 Mendeteksi & Memvisualisasikan Data Bolong
Kita dapat menghitung persentase data bolong per kolom menggunakan Pandas dan diagram visual:

In [ ]:
# 1. Menghitung jumlah nilai kosong per kolom
print("Jumlah Nilai Kosong (NaN) Tiap Kolom:")
print(df.isna().sum())

# 2. Visualisasi Heatmap Data Kosong Sederhana
plt.figure(figsize=(7, 3.5))
sns.heatmap(df.isna(), cmap='viridis', cbar=False, yticklabels=False)
plt.title('Peta Sebaran Nilai Kosong (Garis Kuning = Data Hilang/NaN)', fontweight='bold')
plt.show()

### 🛠️ Strategi 1: Kapan Boleh Menghapus Data (`.dropna()`)?
- **Boleh Dihapus**: Jika baris yang hilang hanya sedikit sekali (< 2% dari total data) atau kolom tersebut hilang lebih dari 70% sehingga tidak berguna.
- **Dilarang Dihapus**: Jika data Anda sedikit atau data yang hilang memiliki informasi berharga. Menghapus data sembarangan akan menyebabkan bias (*sampling bias*)!

### 🛠️ Strategi 2: Imputasi Statistik (Mean, Median, Modus)
- **Median (Nilai Tengah)**: **Paling Direkomendasikan untuk Data Numerik!** Kenapa? Karena Median tahan terhadap angka ekstrem (*outlier*), sedangkan Mean (rata-rata) sangat mudah rusak jika ada angka jutaan atau angka aneh.
- **Modus (*Most Frequent*)**: **Paling Direkomendasikan untuk Data Kategorikal/Teks!** Mengisi dengan kategori yang paling sering muncul.

In [ ]:
from sklearn.impute import SimpleImputer

# 1. Imputasi Numerik (Usia, Gaji, Skor Kredit) dengan MEDIAN
imputer_num = SimpleImputer(strategy='median')
kolom_num = ['Usia', 'Gaji_Juta', 'Skor_Kredit']
df[kolom_num] = imputer_num.fit_transform(df[kolom_num])

# 2. Imputasi Kategorikal (Tingkat Pendidikan, Status Pernikahan) dengan MODUS (most_frequent)
imputer_cat = SimpleImputer(strategy='most_frequent')
kolom_cat = ['Tingkat_Pendidikan', 'Status_Pernikahan']
df[kolom_cat] = imputer_cat.fit_transform(df[kolom_cat])

print("Status Nilai Kosong Setelah Imputasi:")
print(df.isna().sum())  # Seluruh kolom kini 0 (bersih dari NaN)! ✅

---

## 🕵️ 3. Deteksi & Penanganan Pencilan (*Outliers*)

### Apa Itu Outlier?
Outlier adalah data yang nilainya **menyimpang terlalu jauh dari mayoritas data lainnya**.
- Contoh di dataset kita:
  - Kolom `Usia`: Ada nasabah berusia **120 tahun**! Ini jelas salah ketik saat input formulir (mungkin maksudnya 20 tahun).
  - Kolom `Gaji_Juta`: Ada nasabah dengan gaji **Rp 95 Juta**, sementara nasabah lainnya bergaji Rp 5–15 Juta. Angka ini mungkin benar (konglomerat), tetapi bisa merusak model machine learning.

### A. Deteksi Visual: Boxplot
Diagram Boxplot secara otomatis menandai outlier sebagai titik-titik kecil di luar garis kumis (*whiskers*):

In [ ]:
plt.figure(figsize=(10, 3))
plt.subplot(1, 2, 1)
sns.boxplot(x=df['Usia'], color='lightcoral')
plt.title('Deteksi Outlier Usia (Titik di 120 thn!)', fontweight='bold')

plt.subplot(1, 2, 2)
sns.boxplot(x=df['Gaji_Juta'], color='skyblue')
plt.title('Deteksi Outlier Gaji (Titik di 95 Juta!)', fontweight='bold')
plt.tight_layout()
plt.show()

### B. Deteksi Matematis: Metode IQR (*Interquartile Range*)
Metode standar dunia untuk menentukan batas kewajaran data:
1. Hitung Kuartil 1 ($Q1$ = 25%) dan Kuartil 3 ($Q3$ = 75%).
2. Hitung rentang $IQR = Q3 - Q1$.
3. **Batas Bawah** = $Q1 - (1.5 \times IQR)$
4. **Batas Atas** = $Q3 + (1.5 \times IQR)$

Semua angka di luar batas ini resmi dianggap sebagai **Outlier**!

In [ ]:
# Menghitung Batas IQR untuk Kolom Usia
Q1_usia = df['Usia'].quantile(0.25)
Q3_usia = df['Usia'].quantile(0.75)
IQR_usia = Q3_usia - Q1_usia

batas_bawah_usia = Q1_usia - (1.5 * IQR_usia)
batas_atas_usia = Q3_usia + (1.5 * IQR_usia)

print(f"Q1 Usia: {Q1_usia:.1f}, Q3 Usia: {Q3_usia:.1f}, IQR: {IQR_usia:.1f}")
print(f"Batas Wajar Usia: {batas_bawah_usia:.1f} tahun s/d {batas_atas_usia:.1f} tahun")

# Capping / Winsorizing (Membatasi nilai ekstrem ke batas wajar maksimum)
df['Usia_Bersih'] = df['Usia'].clip(lower=batas_bawah_usia, upper=batas_atas_usia)
df['Gaji_Bersih'] = df['Gaji_Juta'].clip(lower=df['Gaji_Juta'].quantile(0.05), upper=df['Gaji_Juta'].quantile(0.90))

print("\nData Usia dan Gaji Setelah Outlier Dibatasi (Capped):")
df[['ID_Nasabah', 'Usia', 'Usia_Bersih', 'Gaji_Juta', 'Gaji_Bersih']].head(6)

---

## ⚖️ 4. Penskalaan Fitur Numerik (*Feature Scaling*)

### Mengapa Scaling Itu Wajib?
Perhatikan dua kolom ini:
- Kolom `Usia`: Rentang angkanya hanya puluhan (**20 – 50**).
- Kolom `Skor_Kredit`: Rentang angkanya ratusan (**500 – 800**).
- Kolom `Gaji`: Rentang angkanya jutaan (**5.000.000 – 15.000.000**).

Algoritma data mining yang berbasis jarak (seperti **K-NN, K-Means Clustering, SVM**) akan mengira bahwa kolom Gaji dan Skor Kredit jauh lebih penting daripada Usia hanya karena nominal angkanya lebih besar!  
👉 **Solusi:** Kita harus menyamakan skala semua kolom agar adil!

### 3 Metode Penskalaan Terpopuler:
| Scaler | Rumus / Cara Kerja | Rentang Hasil | Kapan Dipakai? |
| :--- | :--- | :--- | :--- |
| **MinMaxScaler** | $\frac{X - X_{min}}{X_{max} - X_{min}}$ | Tepat **0.0 sampai 1.0** | Bagus jika data tidak memiliki outlier ekstrem dan algoritma butuh batas pasti (seperti neural network). |
| **StandardScaler** | $\frac{X - \mu}{\sigma}$ (Z-Score) | Rata-rata 0, Std 1 | **Paling Populer & Standar!** Mengubah data menjadi distribusi normal baku. |
| **RobustScaler** | $\frac{X - Median}{IQR}$ | Berbasis kuartil | **Pilihan Terbaik jika data masih mengandung outlier!** |

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

fitur_angka = df[['Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit']]

# 1. MinMaxScaler (Rentang 0 s/d 1)
mms = MinMaxScaler()
data_minmax = mms.fit_transform(fitur_angka)

# 2. StandardScaler (Rata-rata 0, Std 1)
std = StandardScaler()
data_std = std.fit_transform(fitur_angka)

print("=== CONTOH HASIL MINMAX SCALER (0 s/d 1) ===")
print(pd.DataFrame(data_minmax, columns=['Usia_MMS', 'Gaji_MMS', 'Skor_MMS']).head(3))

print("\n=== CONTOH HASIL STANDARD SCALER (Rata-rata 0, Std 1) ===")
print(pd.DataFrame(data_std, columns=['Usia_STD', 'Gaji_STD', 'Skor_STD']).head(3))

---

## 🏷️ 5. Pengkodean Data Kategorikal (*Categorical Encoding*)

Algoritma komputer hanya bisa memproses angka matematika, bukan teks string. Maka kita harus mengubah teks kategori menjadi angka.

Gunakan aturan pemilihan ini:
1. **Ordinal Encoding**: Gunakan jika kategori **memiliki tingkatan/peringkat yang jelas**  
   *Contoh:* Pendidikan: `SMA = 0`, `D3 = 1`, `S1 = 2`, `S2 = 3`.
2. **One-Hot Encoding**: Gunakan jika kategori **tidak memiliki tingkatan (nominal / setara)**  
   *Contoh:* Status Pernikahan: dibuat kolom biner baru `Menikah` (1 jika menikah, 0 jika lajang).  
   *Kiat:* Selalu gunakan parameter `drop_first=True` untuk menghindari jebakan redundansi (*Dummy Variable Trap*).

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# 1. Ordinal Encoding untuk Tingkat Pendidikan (Ada Peringkat!)
hierarki_pendidikan = [['SMA', 'D3', 'S1', 'S2']]
oe = OrdinalEncoder(categories=hierarki_pendidikan)
df['Pendidikan_Encoded'] = oe.fit_transform(df[['Tingkat_Pendidikan']])

# 2. One-Hot Encoding untuk Status Pernikahan (Nominal / Tanpa Peringkat)
df_final = pd.get_dummies(df, columns=['Status_Pernikahan'], drop_first=True, dtype=int)

print("=== DATASET AKHIR YANG BERSIH, TERSTANDARISASI & SIAP DILATIH! ===")
kolom_siap = ['ID_Nasabah', 'Usia_Bersih', 'Gaji_Bersih', 'Skor_Kredit', 'Pendidikan_Encoded', 'Status_Pernikahan_Menikah']
df_final[kolom_siap]

---

## 🧠 6. Kuis Evaluasi Pemahaman Mandiri

Uji pemahaman Anda terhadap 4 pilar data cleaning & preprocessing:

In [ ]:
# Evaluasi Pemahaman Mandiri Modul 2
evaluasi_m2 = [
    {
        'no': 1,
        'soal': 'Mengapa mengisi nilai kosong numerik dengan MEDIAN lebih disarankan daripada MEAN jika data memiliki outlier?',
        'jawaban': 'Karena Median adalah nilai tengah yang kebal terhadap angka ekstrem, sedangkan Mean akan tertarik drastis oleh angka outlier.'
    },
    {
        'no': 2,
        'soal': 'Apa rumus batas atas outlier menggunakan metode IQR?',
        'jawaban': 'Batas Atas = Q3 + (1.5 × IQR). Data di atas angka ini dianggap outlier.'
    },
    {
        'no': 3,
        'soal': 'Mengapa algoritma berbasis jarak seperti K-NN dan K-Means mewajibkan penskalaan fitur (Scaling)?',
        'jawaban': 'Agar fitur dengan rentang angka besar (misal gaji jutaan) tidak mendominasi perhitungan jarak dibanding fitur berangka kecil (misal usia puluhan).'
    },
    {
        'no': 4,
        'soal': 'Kapan kita menggunakan Ordinal Encoding dan kapan menggunakan One-Hot Encoding?',
        'jawaban': 'Ordinal Encoding untuk data yang punya hierarki tingkatan (SD < SMP < S1), sedangkan One-Hot Encoding untuk data nominal setara (Warna: Merah, Biru).'
    }
]

for ev in evaluasi_m2:
    print(f"Soal {ev['no']}: {ev['soal']}")
    print(f"👉 Kunci: {ev['jawaban']}\n")
print("Jika Anda memahami ke-4 pilar di atas, Anda siap menghadapi dataset kotor apapun di dunia nyata! 🌟")

---

## ✍️ 7. Kotak Latihan Bebas (*Playground Challenge*)

Selesaikan tantangan terpandu di bawah ini untuk menguji keahlian baru Anda:

In [ ]:
# TANTANGAN MANDIRI:
# Diberikan data tinggi badan pasien rumah sakit berikut:
tinggi_pasien = pd.Series([160, 165, np.nan, 170, 155, 290, 168]) # Ada 1 NaN dan 1 Outlier ekstrem (290 cm!)

# 1. Isi missing value dengan nilai Median
tinggi_bersih = tinggi_pasien.fillna(tinggi_pasien.median())

# 2. Batasi nilai maksimal wajar di angka 190 cm menggunakan .clip()
tinggi_final = tinggi_bersih.clip(upper=190)

print("Data Tinggi Pasien Sebelum dan Sesudah Dibersihkan:")
print(pd.DataFrame({'Tinggi_Awal': tinggi_pasien, 'Tinggi_Bersih_Final': tinggi_final}))

---

## 🎯 8. Checklist Kelulusan Modul 02 (*Hero Checkpoint*)

Beri tanda centang pada hati Anda jika Anda sudah menguasai poin-poin berikut:
- [ ] Saya paham pepatah *Garbage In, Garbage Out*.
- [ ] Saya tahu perbedaan konsep data hilang: MCAR, MAR, dan MNAR.
- [ ] Saya bisa mengisi missing values dengan Median (angka) dan Modus (kategori).
- [ ] Saya bisa mendeteksi outlier menggunakan grafik Boxplot dan rumus IQR ($Q1 - 1.5 \times IQR$).
- [ ] Saya paham mengapa fitur numerik wajib disetarakan skalanya (*Feature Scaling*).
- [ ] Saya bisa memilih antara MinMaxScaler, StandardScaler, atau RobustScaler.
- [ ] Saya bisa membedakan Ordinal Encoding (bertingkat) dan One-Hot Encoding (nominal setara).

---

## 🏁 Gerbang Menuju Modul 03: EDA & Feature Engineering!

Kerja yang luar biasa! Data Anda kini sudah bersih, rapi, dan memiliki format angka yang adil untuk dilatih oleh komputer.

Langkah berikutnya adalah **mengekstrak wawasan tersembunyi dan menciptakan variabel-variabel baru yang lebih cerdas**:

👉 **[Buka Modul 03: Exploratory Data Analysis & Feature Engineering](03_EDA_and_Feature_Engineering.ipynb)**

Sampai jumpa di Modul 03! Tetap semangat dan jadilah Hero Data Mining! 🚀